# Data cleaning
## Importamos librerías y los archivos csv

In [29]:
import pandas as pd
# Database CPI - Alimentos
grupos = pd.read_csv(r"C:\Users\ASUS\Desktop\CFBPredic\data\ipc.csv")
# Database Alimentos subproductos
productos = pd.read_csv(r"C:\Users\ASUS\Desktop\CFBPredic\data\productos.csv")

In [30]:
# Revisamos tabla CPI alimentos
grupos.head()

,Año,Mes,Grupo,Subgrupo,Indicador_grup,Indicador
0,2006,Enero,Alimentos y No alimentos,Alimentos,60.91,Índice
1,2006,Febrero,Alimentos y No alimentos,Alimentos,62.03,Índice
2,2006,Febrero,Alimentos y No alimentos,Alimentos,1.84,Variación mensual
3,2006,Marzo,Alimentos y No alimentos,Alimentos,63.10,Índice
4,2006,Marzo,Alimentos y No alimentos,Alimentos,1.73,Variación mensual


In [31]:
# Revisamos tabla IPC de subproductos
productos.head()

,Año,Mes,'Series_IPC'[Ciudad],Nivel,Cód. CCIF,Descripción CCIF,'Indicadores_cuboIPC'[Indicador],'Filtro Indicador'[Indicador]
0,2005,Enero,Nacional,Clase,111,Pan y cereales (ND),51.48,Índice
1,2005,Enero,Nacional,Clase,112,Carne (ND),56.89,Índice
2,2005,Enero,Nacional,Clase,113,Pescado (ND),55.78,Índice
3,2005,Enero,Nacional,Clase,114,"Leche, queso y huevos (ND)",62.85,Índice
4,2005,Enero,Nacional,Clase,115,Aceites y grasas (ND),51.42,Índice


## Procesamiento para la tabla grupos y productos

In [32]:
# Manejamos los meses a números representativos tanto para grupos como para productos
map_mes = {
    "Enero": 1, "Febrero": 2, "Marzo": 3, "Abril": 4,
    "Mayo": 5, "Junio": 6, "Julio": 7, "Agosto": 8,
    "Septiembre": 9, "Setiembre": 9,  # por si acaso
    "Octubre": 10, "Noviembre": 11, "Diciembre": 12
}

for df in [grupos, productos]:
    df["mes_num"] = df["Mes"].map(map_mes)
    df["date"] = pd.to_datetime(
        dict(year=df["Año"], month=df["mes_num"], day=1)
    )

In [33]:
# Filtramos alimentos (índice)
alimentos = grupos[
    (grupos["Indicador"] == "Índice")
].copy()

# Seleccionamos solo la fecha y el indicador de alimentos
alimentos = alimentos[["date", "Indicador_grup"]].rename(
    columns={"Indicador_grup": "ipc_alimentos_index"}
).sort_values("date").reset_index(drop=True)

In [34]:
# Revisión del archivo alimentos
alimentos.head()

,date,ipc_alimentos_index
0,2006-01-01,60.91
1,2006-02-01,62.03
2,2006-03-01,63.10
3,2006-04-01,62.45
4,2006-05-01,61.91


In [35]:
# Filtramos productos por Índice y clase
productos = productos[
    (productos["'Filtro Indicador'[Indicador]"] == "Índice") &
    (productos["Nivel"] == "Clase")
].copy()

# Renombramos las columnas para un mejor menejo
productos = productos.rename(columns={
    "Año": "year",
    "Mes": "month",
    "'Series_IPC'[Ciudad]": "ciudad",
    "Nivel": "nivel",
    "Cód. CCIF": "ccif",
    "Descripción CCIF": "descripcion",
    "'Indicadores_cuboIPC'[Indicador]": "valor",
    "'Filtro Indicador'[Indicador]": "tipo_indicador"
})

# Filtramos solo las columnas a usar
productos = productos[["date", "descripcion", "valor"]].copy()

In [36]:
# Revisión del archivo productos
productos.head()

,date,descripcion,valor
0,2005-01-01,Pan y cereales (ND),51.48
1,2005-01-01,Carne (ND),56.89
2,2005-01-01,Pescado (ND),55.78
3,2005-01-01,"Leche, queso y huevos (ND)",62.85
4,2005-01-01,Aceites y grasas (ND),51.42


## Merge productos con ipc

In [37]:
# Crear panel de productos
panel_productos = productos.pivot_table(
    index="date",
    columns="descripcion",
    values="valor"
).reset_index()

In [38]:
# Estandarizamos carácteres especiales
panel_productos.columns = [
    col.lower()
       .replace(" ", "_")
       .replace("-", "_")
       .replace("(nd)", "")
       .replace(".", "_")
       .replace(",", "")
       .replace('"', '')
       .replace("á", "a")
       .replace("é", "e")
       .replace("í", "i")
       .replace("ó", "o")
       .replace("ú", "u")
       .replace("ñ", "n")
    for col in panel_productos.columns
]

In [39]:
# Combinamos los datasets (Alimentos y Productos) por fecha
data = alimentos.merge(panel_productos, on="date", how="inner").sort_values("date")
data.columns = [col.rstrip("_") for col in data.columns]

In [40]:
# Revisamos el dataset final
data.head()

,date,ipc_alimentos_index,aceites_y_grasas,aguas_minerales_refrescos_jugos_de_frutas_y_de_legumbres,azucar_mermelada_miel_chocolate_y_dulces_de_azucar,cafe_te_y_cacao,carne,frutas,leche_queso_y_huevos,legumbres_hortalizas,pan_y_cereales,pescado,productos_alimenticios_n_e_p
0,2006-01-01,60.91,51.33,62.47,55.45,48.95,61.03,63.57,66.73,72.88,50.38,57.44,60.35
1,2006-02-01,62.03,50.93,63.41,56.86,48.23,63.90,66.18,66.12,74.62,50.30,58.12,59.94
2,2006-03-01,63.10,51.28,63.43,58.43,49.08,64.41,69.87,66.05,77.43,50.69,59.30,60.96
3,2006-04-01,62.45,51.24,62.47,60.56,49.07,62.82,67.99,66.55,75.19,50.32,61.94,61.56
4,2006-05-01,61.91,51.36,62.40,64.10,49.06,62.92,66.12,67.07,70.09,50.52,63.21,61.78


In [41]:
# Guardamos el dataset limpio
# data.to_csv(r'C:\Users\ASUS\Desktop\CFBPredic\src\data.csv', 
#                    index=False)